<a href="https://colab.research.google.com/github/neurolawal/Alljoined_prepro/blob/main/pytorch_eeg_emotion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

See Documentation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
num_epochs = 10
batch_size = 32
learning_rate = 0.001

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"samneuro","key":"7484002ba4aaf0460068e25a467ccac6"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d birdy654/eeg-brainwave-dataset-feeling-emotions
!unzip -q eeg-brainwave-dataset-feeling-emotions.zip

Dataset URL: https://www.kaggle.com/datasets/birdy654/eeg-brainwave-dataset-feeling-emotions
License(s): copyright-authors
  0% 0.00/11.9M [00:00<?, ?B/s]
100% 11.9M/11.9M [00:00<00:00, 1.32GB/s]


In [ ]:
df = pd.read_csv('emotions.csv')

In [ ]:
label_mapping = {'NEGATIVE': 0, 'NEUTRAL': 1, 'POSITIVE': 2}
df['label'] = df['label'].map(label_mapping)

In [ ]:
x_data = df.drop('label', axis=1).values
y_data = df['label'].values

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, random_state=42)

In [ ]:
# moving to tensors now
train_dataset = torch.utils.data.TensorDataset(torch.tensor(x_train, dtype=torch.float32),
                                               torch.tensor(y_train, dtype=torch.long))
test_dataset = torch.utils.data.TensorDataset(torch.tensor(x_test, dtype=torch.float32),
                                              torch.tensor(y_test, dtype=torch.long))

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
classes = ('negative', 'neutral', 'positive')

In [ ]:
class EEGNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 2548 features -> Hidden: 512 neurons
        self.fc1 = nn.Linear(2548, 512)
        self.fc2 = nn.Linear(512, 256)
        # Output: 3 classes (Negative, Neutral, Positive)
        self.fc3 = nn.Linear(256, 3)

    def forward(self, x):
        # We use small x as requested
        x = F.relu(self.fc1(x)) # Activation
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = EEGNet().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
n_total_steps = len(train_loader)
for epoch in range(num_epochs):
    running_loss = 0.0

    for i, (x_batch, labels) in enumerate(train_loader):
        x_batch = x_batch.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(x_batch)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad() # Clear BEFORE backward
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'[{epoch + 1}] loss: {running_loss / n_total_steps:.3f}')

print('Finished Training')
torch.save(model.state_dict(), './eeg_model.pth')

[1] loss: 7881432365776.593
[2] loss: 37251879116834.219
[3] loss: 18153656017360.594
[4] loss: 25190856805850.074
[5] loss: 21040334508619.852
[6] loss: 18216021664881.777
[7] loss: 32784984539742.816
[8] loss: 31557551416604.445
[9] loss: 63180485900449.188
[10] loss: 47283282761803.852
Finished Training


In [ ]:
loaded_model = EEGNet()

In [ ]:
loaded_model.load_state_dict(torch.load('./eeg_model.pth'))
loaded_model.to(device)
loaded_model.eval()

EEGNet(
  (fc1): Linear(in_features=2548, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=3, bias=True)
)

In [ ]:
with torch.no_grad(): # Disables gradient tracking to save memory
    n_correct = 0
    n_samples = len(test_loader.dataset)

    for x_batch, labels in test_loader:
        x_batch = x_batch.to(device)
        labels = labels.to(device)

        # Get predictions from the loaded model
        outputs = loaded_model(x_batch)

        # torch.max returns (value, index)
        _, predicted = torch.max(outputs, 1)
        n_correct += (predicted == labels).sum().item()

    acc = 100.0 * n_correct / n_samples
    print(f'Accuracy of the EEG model on the test data: {acc:.2f} %')